### 1. Identify the top 500 movies

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, round

spark = SparkSession.builder.appName("TopMovies").master("local[*]").getOrCreate()

PROJECT = "/Users/fred/Desktop/Projects/MovieRecommender"

df_ratings = spark.read.parquet(f"{PROJECT}/bronze/ratings")
df_links   = spark.read.parquet(f"{PROJECT}/bronze/links")
df_movies  = spark.read.parquet(f"{PROJECT}/bronze/movies")

top_500 = (
    df_ratings
    .groupBy("movieId")
    .agg(count("*").alias("rating_count"), round(avg("rating"), 2).alias("avg_rating"))
    .orderBy(col("rating_count").desc())
    .limit(500)
)

movies_to_scrape = (
    top_500
    .join(df_links, "movieId")
    .join(df_movies, "movieId")
    .select("movieId", "title", "tmdbId", "imdbId", "rating_count", "avg_rating")
    .filter(col("tmdbId").isNotNull())
    .orderBy(col("avg_rating").desc())
    .collect()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/17 12:36:24 WARN Utils: Your hostname, MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.12 instead (on interface en0)
26/02/17 12:36:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/17 12:36:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
for row in movies_to_scrape[:10]:
    print(f"{row.movieId:<10}{row.title:<50} | IMDB: {row.imdbId: <10} | TMDB: {row.tmdbId: <10} | Ratings: {row.rating_count: <8} | Avg: {row.avg_rating}")

print(f"\nTotal movies: {len(movies_to_scrape)}")

318       Shawshank Redemption, The (1994)                   | IMDB: 111161     | TMDB: 278        | Ratings: 102929   | Avg: 4.4
858       Godfather, The (1972)                              | IMDB: 68646      | TMDB: 238        | Ratings: 66440    | Avg: 4.32
50        Usual Suspects, The (1995)                         | IMDB: 114814     | TMDB: 629        | Ratings: 67750    | Avg: 4.27
1203      12 Angry Men (1957)                                | IMDB: 50083      | TMDB: 389        | Ratings: 21863    | Avg: 4.27
1221      Godfather: Part II, The (1974)                     | IMDB: 71562      | TMDB: 240        | Ratings: 43111    | Avg: 4.26
2019      Seven Samurai (Shichinin no samurai) (1954)        | IMDB: 47478      | TMDB: 346        | Ratings: 16531    | Avg: 4.25
527       Schindler's List (1993)                            | IMDB: 108052     | TMDB: 424        | Ratings: 73849    | Avg: 4.24
904       Rear Window (1954)                                 | IMDB: 47396      | TM

### 2. Write the scraper utility

In [3]:
import os
import time
import requests
from dotenv import load_dotenv

load_dotenv()

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {os.getenv('TMDB_BEARER_TOKEN')}",
    "accept": "application/json",
})


def fetch_movie(tmdb_id: int) -> dict | None:
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?append_to_response=credits"
    while True:
        resp = session.get(url, timeout=10)
        if resp.status_code == 429:
            time.sleep(int(resp.headers.get("Retry-After", 2)))
            continue
        if resp.status_code != 200:
            print(f"  WARN: tmdb_id={tmdb_id} -> {resp.status_code}")
            return None
        return resp.json()


def extract(raw: dict, movie_id: int) -> dict:
    poster = raw.get("poster_path")
    return {
        "movieId": movie_id,
        "tmdbId": raw.get("id"),
        "title": raw.get("title"),
        "directors": [m["name"] for m in raw.get("credits", {}).get("crew", []) if m.get("job") == "Director"],
        "budget": raw.get("budget"),
        "revenue": raw.get("revenue"),
        "runtime": raw.get("runtime"),
        "release_date": raw.get("release_date"),
        "poster_url": f"https://image.tmdb.org/t/p/w500{poster}" if poster else None,
        "overview": raw.get("overview"),
        "vote_average": raw.get("vote_average"),
        "original_language": raw.get("original_language"),
    }

### 3. Scrape & save to parquet

In [4]:
import json
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import *

results = []
total = len(movies_to_scrape)

for i, row in enumerate(movies_to_scrape):
    print(f"[{i+1}/{total}] tmdbId={row.tmdbId} ({row.title})")
    raw = fetch_movie(row.tmdbId)
    if raw:
        results.append(extract(raw, row.movieId))
    time.sleep(0.05)

print(f"\nScraped {len(results)}/{total} movies.")

# Save JSON for sharing
json_path = f"{PROJECT}/bronze/enrichment/scraped_metadata.json"
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"JSON saved to {json_path}")

# Save parquet for pipeline
schema = StructType([
    StructField("movieId", IntegerType()),
    StructField("tmdbId", IntegerType()),
    StructField("title", StringType()),
    StructField("directors", ArrayType(StringType())),
    StructField("budget", LongType()),
    StructField("revenue", LongType()),
    StructField("runtime", IntegerType()),
    StructField("release_date", StringType()),
    StructField("poster_url", StringType()),
    StructField("overview", StringType()),
    StructField("vote_average", FloatType()),
    StructField("original_language", StringType()),
])

df_enrichment = (
    spark.createDataFrame(results, schema)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("tmdb_api"))
)

df_enrichment.write.mode("overwrite").parquet(f"{PROJECT}/bronze/enrichment/parquet")

print(f"Done. {df_enrichment.count()} rows written to bronze/enrichment/parquet")
df_enrichment.show(5, truncate=False)

[1/500] tmdbId=278 (Shawshank Redemption, The (1994))
[2/500] tmdbId=238 (Godfather, The (1972))
[3/500] tmdbId=629 (Usual Suspects, The (1995))
[4/500] tmdbId=389 (12 Angry Men (1957))
[5/500] tmdbId=240 (Godfather: Part II, The (1974))
[6/500] tmdbId=346 (Seven Samurai (Shichinin no samurai) (1954))
[7/500] tmdbId=424 (Schindler's List (1993))
[8/500] tmdbId=567 (Rear Window (1954))
[9/500] tmdbId=550 (Fight Club (1999))
[10/500] tmdbId=129 (Spirited Away (Sen to Chihiro no kamikakushi) (2001))
[11/500] tmdbId=680 (Pulp Fiction (1994))
[12/500] tmdbId=935 (Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964))
[13/500] tmdbId=510 (One Flew Over the Cuckoo's Nest (1975))
[14/500] tmdbId=213 (North by Northwest (1959))
[15/500] tmdbId=289 (Casablanca (1942))
[16/500] tmdbId=769 (Goodfellas (1990))
[17/500] tmdbId=598 (City of God (Cidade de Deus) (2002))
[18/500] tmdbId=155 (Dark Knight, The (2008))
[19/500] tmdbId=603 (Matrix, The (1999))
[20/500] tmdbId=128 (Pri

26/02/17 12:37:48 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/02/17 12:37:48 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/02/17 12:37:48 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 76,00% for 10 writers
26/02/17 12:37:48 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 69,09% for 11 writers
26/02/17 12:37:48 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 63,33% for 12 writers
26/02/17 12:37:48 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 69,09% for 11 writers
26/02/17 12:37:48 WARN MemoryManager: Total allocation exceeds 95,